# TP PAC-Bayes -- Session 3: Beyond forests: Gibbs posteriors, hold-out, imbalance, boosting

Solution notebook for Session 3 (three hours, Parts 3.1 to 3.4 of the sheet `tp_pacbayes.pdf`). It reproduces the numbers, tables (written to `results/`) and figures (written to `figures/`) of the solution sheet `tp_pacbayes_solutions.pdf`.

It needs `pbtools.py`, `prepdata.py` and the folder `datasets/` of the course, either in this folder or in its parent folder. Each session notebook is self-contained: the helper functions it needs are redefined at the top.

In [1]:
import os, sys, time
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler

HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
for p in [HERE, os.path.join(HERE, ".."), os.path.join(HERE, "..", "data"), os.path.join(HERE, "..", "..", "common"), os.path.join(HERE, "data"), os.path.join(HERE, "..", "common")]:
    if p not in sys.path:
        sys.path.insert(0, p)
from pbtools import (kl_bin, kl_inv_upper, kl_inv_lower, kl_div, bound_seeger, bound_mcallester,
                     BaggedTrees, mv_statistics, fo_bound, tnd_bound, cbound_pb, softmax,
                     optimize_fo, optimize_tnd)
from prepdata import data_recovery

DATADIR = next((d for d in [os.path.join(HERE, "data"), HERE, os.path.join(HERE, ".."), os.path.join(HERE, "..", "data")] if os.path.isdir(os.path.join(d, "datasets"))), HERE)
RES = os.path.join(HERE, "results"); FIG = os.path.join(HERE, "figures")
os.makedirs(RES, exist_ok=True); os.makedirs(FIG, exist_ok=True)
DELTA = 0.05
N_SPLITS = 5
BALANCED = ["australian", "balance", "bupa", "german", "heart", "iono", "pima", "sonar", "wdbc", "spambase"]
IMBALANCED = ["abalone8", "pageblocks", "satimage", "segmentation", "yeast3"]
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False})


def load(name):
    """Load a dataset of the course with labels in {-1,+1} and standardized features."""
    cwd = os.getcwd(); os.chdir(DATADIR)
    try:
        X, y = data_recovery(name)
    finally:
        os.chdir(cwd)
    X = StandardScaler().fit_transform(np.asarray(X, dtype=float))
    return X, np.where(np.asarray(y) > 0, 1, -1)


def eff(rho):
    """Effective number of voters 1/sum rho^2."""
    return 1.0 / np.sum(rho ** 2)


def save_tab(name, rows):
    open(os.path.join(RES, name), "w").write("\n".join(rows) + "\n")
    print("  -> results/" + name)


def ms(a, d=3):
    a = np.asarray(a, dtype=float)
    return f"{a.mean():.{d}f} ({a.std():.{d}f})"

## Helpers from Session 1 (random forests with out-of-bag statistics)

In [2]:
def forest_stats(X, y, seed, n_trees=100, frac=0.5, max_features="sqrt", max_depth=None):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=seed)
    bt = BaggedTrees(n_trees, bootstrap_frac=frac, max_features=max_features, max_depth=max_depth,
                     random_state=seed).fit(Xtr, ytr)
    L, n1 = bt.oob_losses(Xtr, ytr); Lt, n2 = bt.oob_tandem(Xtr, ytr); D, n3 = bt.oob_disagreement(Xtr)
    return dict(bt=bt, Xtr=Xtr, Xte=Xte, ytr=ytr, yte=yte, L=L, n1=n1, Lt=Lt, n2=n2, D=D, n3=n3,
                Vte=bt.predict_all(Xte), pi=np.ones(n_trees) / n_trees)


def certificates(F, rho):
    return (fo_bound(rho, F["pi"], F["L"], F["n1"], DELTA), tnd_bound(rho, F["pi"], F["Lt"], F["n2"], DELTA),
            cbound_pb(rho, F["pi"], F["L"], F["n1"], F["D"], F["n3"], DELTA))


CACHE = {}


def part2_3():
    print("Part 2 and 3")
    rows2, rows3, summary = [], [], {k: [] for k in ["u", "fo", "tnd"]}
    for name in BALANCED + IMBALANCED:
        X, y = load(name)
        R = {k: [] for k in ["mv", "gibbs", "dis", "fo", "tnd", "cb", "rf", "n1", "n2",
                             "mv_fo", "cert_fo", "eff_fo", "mv_tnd", "cert_tnd", "eff_tnd", "best_u"]}
        for seed in range(N_SPLITS):
            F = forest_stats(X, y, seed)
            CACHE[(name, seed)] = F
            st = mv_statistics(F["Vte"], F["yte"], F["pi"])
            fo, tnd, cb = certificates(F, F["pi"])
            rf = RandomForestClassifier(100, random_state=seed).fit(F["Xtr"], F["ytr"])
            for k, v in [("mv", st["mv"]), ("gibbs", st["gibbs"]), ("dis", st["dis"]), ("fo", fo), ("tnd", tnd),
                         ("cb", cb), ("rf", np.mean(rf.predict(F["Xte"]) != F["yte"])), ("n1", F["n1"]), ("n2", F["n2"])]:
                R[k].append(v)
            # Part 3: learning the weights
            r_fo = optimize_fo(F["pi"], F["L"], F["n1"], DELTA)
            r_tnd = optimize_tnd(F["pi"], F["Lt"], F["n2"], DELTA)
            R["mv_fo"].append(mv_statistics(F["Vte"], F["yte"], r_fo)["mv"])
            R["cert_fo"].append(fo_bound(r_fo, F["pi"], F["L"], F["n1"], DELTA)); R["eff_fo"].append(eff(r_fo))
            R["mv_tnd"].append(mv_statistics(F["Vte"], F["yte"], r_tnd)["mv"])
            R["cert_tnd"].append(tnd_bound(r_tnd, F["pi"], F["Lt"], F["n2"], DELTA)); R["eff_tnd"].append(eff(r_tnd))
            # best of the three certificates of the uniform vote, with a union bound (delta/3)
            R["best_u"].append(min(fo_bound(F["pi"], F["pi"], F["L"], F["n1"], DELTA / 3),
                                   tnd_bound(F["pi"], F["pi"], F["Lt"], F["n2"], DELTA / 3),
                                   cbound_pb(F["pi"], F["pi"], F["L"], F["n1"], F["D"], F["n3"], DELTA / 3)))
        M = {k: np.mean(v) for k, v in R.items()}
        rows2.append(f"{name} & {M['rf']:.3f} & {M['mv']:.3f} & {M['gibbs']:.3f} & {M['dis']:.3f} & {int(M['n1'])} & {int(M['n2'])} & "
                     f"{M['fo']:.3f} & {M['tnd']:.3f} & {M['cb']:.3f} \\\\")
        rows3.append(f"{name} & {ms(R['mv'])} & {ms(R['best_u'])} & {ms(R['mv_fo'])} & {ms(R['cert_fo'])} & {np.mean(R['eff_fo']):.1f} & "
                     f"{ms(R['mv_tnd'])} & {ms(R['cert_tnd'])} & {np.mean(R['eff_tnd']):.1f} \\\\")
        summary["u"].append((M["mv"], M["best_u"])); summary["fo"].append((M["mv_fo"], M["cert_fo"])); summary["tnd"].append((M["mv_tnd"], M["cert_tnd"]))
        print(f"  {name}: mv {M['mv']:.3f} FO {M['fo']:.3f} TND {M['tnd']:.3f} CB {M['cb']:.3f} | FO-opt {M['mv_fo']:.3f}/{M['cert_fo']:.3f} TND-opt {M['mv_tnd']:.3f}/{M['cert_tnd']:.3f}")
    s = {k: np.mean(v, axis=0) for k, v in summary.items()}
    rows3.append(r"\midrule")
    rows3.append(f"mean & {s['u'][0]:.3f} & {s['u'][1]:.3f} & {s['fo'][0]:.3f} & {s['fo'][1]:.3f} & & {s['tnd'][0]:.3f} & {s['tnd'][1]:.3f} & \\\\")
    save_tab("p2_forest.tex", rows2); save_tab("p3_weights.tex", rows3)

## Part 3.1 -- The Gibbs posterior on decision stumps (temperature)

In [3]:
def stumps(Xp):
    V = []
    for j in range(Xp.shape[1]):
        for t in np.unique(np.quantile(Xp[:, j], [0.1, 0.25, 0.5, 0.75, 0.9])):
            for s in (1, -1):
                V.append((j, t, s))
    return V


def stump_votes(V, X):
    return np.array([s * np.where(X[:, j] > t, 1, -1) for j, t, s in V])


def part4():
    print("Part 4")
    rows = []
    fig, axes = plt.subplots(1, 2, figsize=(9, 3))
    for name, ax in [("wdbc", axes[0]), ("australian", axes[1])]:
        X, y = load(name)
        Xp, Xr, yp, yr = train_test_split(X, y, train_size=0.2, stratify=y, random_state=0)
        XS, Xt, yS, yt = train_test_split(Xr, yr, test_size=0.4, stratify=yr, random_state=0)
        V = stumps(Xp); VS = stump_votes(V, XS); Vt = stump_votes(V, Xt)
        n, m = len(V), len(yS); P = np.ones(n) / n
        emp = (VS != yS).mean(1); tst = (Vt != yt).mean(1)
        lams = np.logspace(-3, 1.5, 60); B, G, K = [], [], []
        for lam in lams:
            rho = softmax(np.log(P) - lam * m * emp)
            B.append(bound_seeger(rho @ emp, kl_div(rho, P), m, DELTA)); G.append(rho @ emp); K.append(kl_div(rho, P) / m)
        i = int(np.argmin(B)); lam = lams[i]; rho = softmax(np.log(P) - lam * m * emp)
        mv = np.mean(np.where(rho @ Vt >= 0, 1, -1) != yt)
        erm = int(np.argmin(emp)); occ = bound_seeger(emp[erm], np.log(n), m, DELTA)
        rows.append(f"{name} & {n} & {m} & {lam:.3f} & {G[i]:.3f} & {kl_div(rho, P):.2f} & {B[i]:.3f} & {rho @ tst:.3f} & {mv:.3f} & {occ:.3f} & {tst[erm]:.3f} \\\\")
        ax.plot(lams, G, label="empirical Gibbs risk"); ax.plot(lams, K, "--", label="KL/m"); ax.plot(lams, B, lw=2, label="Seeger bound")
        ax.scatter([lam], [B[i]], color="k", zorder=5); ax.set_xscale("log"); ax.set_ylim(0, 0.7)
        ax.set_xlabel(r"$\lambda$"); ax.set_title(f"{name}: {n} stumps, m={m}"); ax.legend(fontsize=7)
    fig.tight_layout(); fig.savefig(os.path.join(FIG, "tp_temperature.pdf")); plt.close(fig)
    save_tab("p4_stumps.tex", rows)

In [4]:
t0 = time.time(); part4(); print(f'{time.time() - t0:.0f} s')

Part 4


  -> results/p4_stumps.tex
2 s


## Part 3.2 -- Hold-out certificate versus self-bounding certificate

In [5]:
def partC():
    print("Part C (hold-out vs self-bounding)")
    rows = []
    for name in ["spambase", "satimage", "pageblocks", "abalone8", "wdbc", "german"]:
        X, y = load(name); R = []
        for seed in range(3):
            Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=seed)
            bt = BaggedTrees(100, bootstrap_frac=0.5, random_state=seed).fit(Xtr, ytr)
            Lt, n2 = bt.oob_tandem(Xtr, ytr); pi = np.ones(100) / 100
            rho = optimize_tnd(pi, Lt, n2, DELTA)
            sb_risk = np.mean(bt.predict(Xte, rho) != yte); sb_cert = tnd_bound(rho, pi, Lt, n2, DELTA)
            X1, X2, y1, y2 = train_test_split(Xtr, ytr, test_size=0.3, stratify=ytr, random_state=seed)
            b = BaggedTrees(100, bootstrap_frac=0.5, random_state=seed).fit(X1, y1)
            ho_risk = np.mean(b.predict(Xte) != yte); e2 = np.mean(b.predict(X2) != y2)
            ho_cert = kl_inv_upper(e2, np.log(1 / DELTA) / len(y2))
            R.append((len(ytr), sb_risk, sb_cert, ho_risk, ho_cert))
        R = np.mean(R, 0)
        rows.append(f"{name} & {int(R[0])} & {R[1]:.3f} & {R[2]:.3f} & {R[3]:.3f} & {R[4]:.3f} \\\\")
    save_tab("pC_holdout.tex", rows)

In [6]:
t0 = time.time(); partC(); print(f'{time.time() - t0:.0f} s')

Part C (hold-out vs self-bounding)


  -> results/pC_holdout.tex
38 s


## Part 3.3 -- Imbalanced datasets and a certificate on the balanced error

In [7]:
def class_restricted(F, cls):
    """OOB statistics restricted to the training examples of one class."""
    bt, Xtr, ytr = F["bt"], F["Xtr"], F["ytr"]
    mask = ytr == cls
    oob_saved = bt.oob_
    bt.oob_ = oob_saved[:, mask]
    Xc, yc = Xtr[mask], ytr[mask]
    L, n1 = bt.oob_losses(Xc, yc); Lt, n2 = bt.oob_tandem(Xc, yc)
    bt.oob_ = oob_saved
    return L, n1, Lt, n2


def part7():
    print("Part 7")
    rows = []
    for name in IMBALANCED:
        R = []
        X, y = load(name)
        for seed in range(N_SPLITS):
            if (name, seed) not in CACHE:          # forests of Part 1.3 (recomputed if needed)
                CACHE[(name, seed)] = forest_stats(X, y, seed)
            F = CACHE[(name, seed)]; pi = F["pi"]
            triv = np.mean(F["yte"] == 1)  # predicting always -1 errs on the positives
            st = mv_statistics(F["Vte"], F["yte"], pi)
            best = min(fo_bound(pi, pi, F["L"], F["n1"], DELTA / 2), tnd_bound(pi, pi, F["Lt"], F["n2"], DELTA / 2))
            # balanced error: one certificate per class with delta/4 each (2 classes x 2 bounds)
            cert_cls, err_cls = [], []
            for cls in (1, -1):
                L, n1, Lt, n2 = class_restricted(F, cls)
                cert_cls.append(min(fo_bound(pi, pi, L, n1, DELTA / 4), tnd_bound(pi, pi, Lt, n2, DELTA / 4)))
                mk = F["yte"] == cls
                err_cls.append(np.mean(np.where(pi @ F["Vte"][:, mk] >= 0, 1, -1) != cls))
            R.append((triv, st["mv"], best, np.mean(err_cls), np.mean(cert_cls), err_cls[0], cert_cls[0]))
        R = np.mean(R, 0)
        rows.append(f"{name} & {R[0]:.3f} & {R[1]:.3f} & {R[2]:.3f} & {R[5]:.3f} & {R[6]:.3f} & {R[3]:.3f} & {R[4]:.3f} \\\\")
    save_tab("p7_imbalanced.tex", rows)

In [8]:
t0 = time.time(); part7(); print(f'{time.time() - t0:.0f} s')

Part 7


  -> results/p7_imbalanced.tex
51 s


## Part 3.4 -- Boosting

In [9]:
def part8():
    print("Part 8")
    rows = []
    for name in ["australian", "german", "pima", "spambase", "wdbc"]:
        X, y = load(name); R = []
        for seed in range(N_SPLITS):
            Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=seed)
            X1, X2, y1, y2 = train_test_split(Xtr, ytr, test_size=0.5, stratify=ytr, random_state=seed)
            ada = AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=100, random_state=seed).fit(X1, y1)
            H = ada.estimators_; T = len(H)
            alpha = np.maximum(ada.estimator_weights_[:T], 1e-12)
            V2 = np.array([h.predict(X2) for h in H]); Vt = np.array([h.predict(Xte) for h in H])
            E2 = (V2 != y2).astype(float); L = E2.mean(1); Lt = E2 @ E2.T / len(y2); n = len(y2)
            pi = np.ones(T) / T; r_ada = alpha / alpha.sum()
            r_fo = optimize_fo(pi, L, n, DELTA); r_tnd = optimize_tnd(pi, Lt, n, DELTA)
            ada_full = AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=100, random_state=seed).fit(Xtr, ytr)
            R.append((np.mean(ada_full.predict(Xte) != yte), mv_statistics(Vt, yte, r_ada)["mv"], r_ada @ L, kl_div(r_ada, pi),
                      fo_bound(r_ada, pi, L, n, DELTA / 2), tnd_bound(r_ada, pi, Lt, n, DELTA / 2),
                      mv_statistics(Vt, yte, r_tnd)["mv"], tnd_bound(r_tnd, pi, Lt, n, DELTA), mv_statistics(Vt, yte, r_fo)["mv"],
                      fo_bound(r_fo, pi, L, n, DELTA), T))
        R = np.mean(R, 0)
        rows.append(f"{name} & {R[0]:.3f} & {R[1]:.3f} & {R[2]:.3f} & {R[3]:.2f} & {R[4]:.3f} & {R[5]:.3f} & {R[8]:.3f} & {R[9]:.3f} & {R[6]:.3f} & {R[7]:.3f} \\\\")
        print("  ", rows[-1])
    save_tab("p8_boosting.tex", rows)

In [10]:
t0 = time.time(); part8(); print(f'{time.time() - t0:.0f} s')

Part 8


   australian & 0.148 & 0.150 & 0.430 & 0.30 & 1.000 & 1.000 & 0.147 & 0.541 & 0.147 & 0.908 \\


   german & 0.245 & 0.242 & 0.453 & 0.39 & 1.000 & 1.000 & 0.301 & 0.782 & 0.287 & 1.000 \\


   pima & 0.257 & 0.263 & 0.443 & 0.29 & 1.000 & 1.000 & 0.268 & 0.789 & 0.247 & 1.000 \\


   spambase & 0.064 & 0.065 & 0.399 & 0.29 & 0.900 & 0.821 & 0.204 & 0.526 & 0.104 & 0.561 \\


   wdbc & 0.043 & 0.043 & 0.265 & 0.03 & 0.783 & 0.709 & 0.103 & 0.332 & 0.063 & 0.465 \\
  -> results/p8_boosting.tex
59 s
